# Stroke Prediction — End-to-End Supervised ML Project

Dataset: **Stroke Prediction Dataset** (Kaggle, by fedesoriano) — file `healthcare-dataset-stroke-data.csv`
Download: https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset

**Yeh project har training concept cover karta hai:**

| Concept | Section |
|---|---|
| EDA | 1 |
| ETL / Cleaning | 2 |
| Feature Engineering (encoding, scaling, impute) | 3 |
| Linear Regression | 4 (predict `bmi`) |
| Train/Test split (stratified) | 5 |
| Logistic, KNN, Naive Bayes, Decision Tree, Random Forest, SVM | 6 |
| Model Evaluation (accuracy, precision, recall, F1, ROC-AUC) | 6 |
| Class Imbalance handling | 7 |
| Model Tuning (GridSearchCV) | 8 |
| Ensemble — Bagging, Boosting, Stacking | 9 |
| Final comparison + best model | 10 |

> Note: Unsupervised intentionally skip kiya hai. Target `stroke` heavily imbalanced hai (~5% positive), isliye **accuracy pe bharosa mat karna — recall/F1 dekhna.**

## Section 0 — Setup & Data Load

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

: 

In [ ]:
# CSV ko notebook ke saath same folder me rakho (Colab: upload karke)
df = pd.read_csv("healthcare-dataset-stroke-data.csv")
print("Shape:", df.shape)
df.head()

: 

## Section 1 — EDA (Exploratory Data Analysis)
Data ko samajhna: types, missing values, distributions, aur sabse important — **target imbalance.**

In [ ]:
df.info()
print("\n--- Numeric summary ---")
df.describe()

In [ ]:
# Missing values
print("Missing values:\n", df.isnull().sum())
print("\nMissing % :\n", (df.isnull().mean()*100).round(2))

In [ ]:
# Target distribution -> imbalance dekho
print(df['stroke'].value_counts())
print("\nStroke rate:", round(df['stroke'].mean()*100, 2), "%")

plt.figure(figsize=(5,3))
sns.countplot(x='stroke', data=df)
plt.title("Target: stroke (0 = no, 1 = yes) — heavily imbalanced")
plt.show()

In [ ]:
# Numeric distributions + relation with target
num_cols = ['age', 'avg_glucose_level', 'bmi']
fig, ax = plt.subplots(1, 3, figsize=(15,4))
for i, c in enumerate(num_cols):
    sns.kdeplot(data=df, x=c, hue='stroke', common_norm=False, ax=ax[i])
    ax[i].set_title(f"{c} by stroke")
plt.tight_layout(); plt.show()

In [ ]:
# Correlation (numeric only)
plt.figure(figsize=(6,4))
sns.heatmap(df[num_cols + ['hypertension','heart_disease','stroke']].corr(),
            annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap"); plt.show()

In [ ]:
# Categorical vs target (quick look)
cat_cols = ['gender','ever_married','work_type','Residence_type','smoking_status']
for c in cat_cols:
    print(f"\n{c} -> stroke rate:")
    print((df.groupby(c)['stroke'].mean()*100).round(2))

## Section 2 — ETL / Cleaning
- `id` drop karo (no predictive value).
- `bmi` me missing values hain → pipeline me median se impute karenge (yahan abhi drop nahi karte).
- `gender` me 1 'Other' value hai — chhod sakte ho, OneHotEncoder handle kar lega.

In [ ]:
df = df.drop(columns=['id'])

# (optional) 'Other' gender ka sirf 1 row hota hai
print("Gender counts:\n", df['gender'].value_counts())
print("\nClean shape:", df.shape)

## Section 3 — Feature Engineering & Preprocessing
`ColumnTransformer` se ek hi jagah:
- **Numeric** (`age`, `avg_glucose_level`, `bmi`) → median impute + StandardScaler (KNN/SVM ke liye scaling zaroori).
- **Categorical** → most-frequent impute + OneHotEncode.
- **Binary** (`hypertension`, `heart_disease`) → as-is passthrough.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['age', 'avg_glucose_level', 'bmi']
categorical_features = ['gender','ever_married','work_type','Residence_type','smoking_status']
binary_features = ['hypertension','heart_disease']

numeric_tf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_tf = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_tf, numeric_features),
    ('cat', categorical_tf, categorical_features),
    ('bin', 'passthrough', binary_features)
])
preprocessor

## Section 4 — Linear Regression (concept cover)
Classification se pehle ek chhota regression detour: **`bmi` predict karo** baaki features se.
Yahan target continuous hai isliye Linear Regression use hota hai. Metrics: R², RMSE.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# bmi ke known rows hi regression ke liye (NaN target nahi chahiye)
reg_df = df[df['bmi'].notna()].copy()
reg_features = ['age','avg_glucose_level','gender','ever_married','work_type',
                'Residence_type','smoking_status','hypertension','heart_disease']
Xr = reg_df[reg_features]
yr = reg_df['bmi']

# bmi ko features se hata diya, isliye chhota preprocessor banate hain
reg_pre = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),
     ['age','avg_glucose_level']),
    ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),
                      ('ohe',OneHotEncoder(handle_unknown='ignore'))]),
     ['gender','ever_married','work_type','Residence_type','smoking_status']),
    ('bin','passthrough',['hypertension','heart_disease'])
])

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=42)
lin = Pipeline([('pre', reg_pre), ('model', LinearRegression())])
lin.fit(Xr_tr, yr_tr)
pred = lin.predict(Xr_te)

print("R2  :", round(r2_score(yr_te, pred), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(yr_te, pred)), 3))
# Note: R2 modest aayega — yeh realistic hai, concept samajhne ke liye theek.

## Section 5 — Classification Setup (Stratified Split)
Ab asli kaam: `stroke` predict karna. Imbalance ki wajah se **stratify** zaroori — warna test set me positive rows kam pad sakte hain.

In [ ]:
X = df.drop(columns=['stroke'])
y = df['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train stroke rate:", round(y_train.mean()*100,2), "%")
print("Test  stroke rate:", round(y_test.mean()*100,2), "%")

## Section 6 — Baseline Models + Evaluation
Sab algorithms ek hi pipeline pattern me: **Logistic, KNN, Naive Bayes, Decision Tree, Random Forest, SVM.**
Har ek ke liye accuracy / precision / recall / F1 / ROC-AUC nikalenge.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)

def evaluate(name, pipe, X_te, y_te):
    y_pred = pipe.predict(X_te)
    # ROC-AUC ke liye probability
    if hasattr(pipe, "predict_proba"):
        y_score = pipe.predict_proba(X_te)[:,1]
    else:
        y_score = pipe.decision_function(X_te)
    return {
        "Model": name,
        "Accuracy":  round(accuracy_score(y_te, y_pred), 3),
        "Precision": round(precision_score(y_te, y_pred, zero_division=0), 3),
        "Recall":    round(recall_score(y_te, y_pred), 3),
        "F1":        round(f1_score(y_te, y_pred), 3),
        "ROC_AUC":   round(roc_auc_score(y_te, y_score), 3),
    }

def make_pipe(clf):
    return Pipeline([('pre', preprocessor), ('clf', clf)])

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "KNN":               KNeighborsClassifier(),
    "NaiveBayes":        GaussianNB(),
    "DecisionTree":      DecisionTreeClassifier(random_state=42),
    "RandomForest":      RandomForestClassifier(random_state=42),
    "SVM":               SVC(probability=True, random_state=42),
}

results = []
for name, clf in models.items():
    pipe = make_pipe(clf)
    pipe.fit(X_train, y_train)
    results.append(evaluate(name, pipe, X_test, y_test))

baseline = pd.DataFrame(results).sort_values("Recall", ascending=False).reset_index(drop=True)
baseline

**Dekho kya ho raha hai:** Accuracy ~95% dikhega kyunki model sab ko "no stroke" bol deta hai (95% wahi hai!). Lekin **Recall ~0** — matlab asli stroke patients miss ho rahe hain. Healthcare me yeh khatarnak hai. Isiliye next section me imbalance handle karenge.

## Section 7 — Class Imbalance Handling
`class_weight='balanced'` minority class ko zyada importance deta hai → recall improve hota hai.
(KNN aur Naive Bayes class_weight support nahi karte — unke liye SMAOTE wala optional cell neeche hai.)

In [ ]:
bal_models = {
    "LogReg_bal": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "DT_bal":     DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "RF_bal":     RandomForestClassifier(random_state=42, class_weight='balanced'),
    "SVM_bal":    SVC(probability=True, random_state=42, class_weight='balanced'),
}

bal_results = []
for name, clf in bal_models.items():
    pipe = make_pipe(clf)
    pipe.fit(X_train, y_train)
    bal_results.append(evaluate(name, pipe, X_test, y_test))

pd.DataFrame(bal_results).sort_values("Recall", ascending=False).reset_index(drop=True)
# Recall ab kaafi badhega (Accuracy thoda girega) — yahi trade-off hai.

**Optional — SMOTE (oversampling):** KNN/NB ke liye useful. Install: `pip install imbalanced-learn`
```python
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
smote_pipe = ImbPipeline([('pre', preprocessor),
                          ('smote', SMOTE(random_state=42)),
                          ('clf', KNeighborsClassifier())])
smote_pipe.fit(X_train, y_train)
```

## Section 8 — Model Tuning (GridSearchCV)
Best dikhne wale model (RandomForest) ko tune karte hain. **Scoring `f1`** rakhenge, accuracy nahi — kyunki imbalance.

In [ ]:
from sklearn.model_selection import GridSearchCV

rf_pipe = make_pipe(RandomForestClassifier(random_state=42, class_weight='balanced'))
param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5],
}
grid = GridSearchCV(rf_pipe, param_grid, scoring='f1', cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV F1 :", round(grid.best_score_, 3))
best_rf = grid.best_estimator_
evaluate("RF_tuned", best_rf, X_test, y_test)

## Section 9 — Ensemble Learning
- **Bagging** — multiple Decision Trees parallel (RandomForest bhi bagging hi hai).
- **Boosting** — sequential, errors fix karte jaana (AdaBoost / GradientBoosting; XGBoost optional).
- **Stacking** — alag models ka output ek meta-model me daalna.

In [ ]:
from sklearn.ensemble import (BaggingClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, StackingClassifier)

ens_results = []

# 1) Bagging
bag = make_pipe(BaggingClassifier(
        estimator=DecisionTreeClassifier(class_weight='balanced', random_state=42),
        n_estimators=100, random_state=42))
bag.fit(X_train, y_train)
ens_results.append(evaluate("Bagging", bag, X_test, y_test))

# 2) Boosting - AdaBoost
ada = make_pipe(AdaBoostClassifier(n_estimators=200, random_state=42))
ada.fit(X_train, y_train)
ens_results.append(evaluate("AdaBoost", ada, X_test, y_test))

# 2b) Boosting - GradientBoosting
gb = make_pipe(GradientBoostingClassifier(random_state=42))
gb.fit(X_train, y_train)
ens_results.append(evaluate("GradientBoosting", gb, X_test, y_test))

# 3) Stacking - KNN + SVM + DT  ->  meta: LogisticRegression
stack = make_pipe(StackingClassifier(
    estimators=[
        ('knn', KNeighborsClassifier()),
        ('svm', SVC(probability=True, class_weight='balanced', random_state=42)),
        ('dt',  DecisionTreeClassifier(class_weight='balanced', random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5))
stack.fit(X_train, y_train)
ens_results.append(evaluate("Stacking", stack, X_test, y_test))

pd.DataFrame(ens_results).sort_values("F1", ascending=False).reset_index(drop=True)

**Optional — XGBoost** (`pip install xgboost`):
```python
from xgboost import XGBClassifier
scale = (y_train==0).sum() / (y_train==1).sum()   # imbalance handle
xgb = make_pipe(XGBClassifier(scale_pos_weight=scale, eval_metric='logloss', random_state=42))
xgb.fit(X_train, y_train)
evaluate("XGBoost", xgb, X_test, y_test)
```

## Section 10 — Final Comparison + Best Model Analysis
Sab results ek table me, fir best model ka confusion matrix aur ROC curve.

In [ ]:
all_results = pd.DataFrame(results + bal_results + ens_results +
                          [evaluate("RF_tuned", best_rf, X_test, y_test)])
all_results = all_results.sort_values("F1", ascending=False).reset_index(drop=True)
all_results

In [ ]:
# Best model (F1 ke hisaab se) ka detailed report
best_name = all_results.iloc[0]['Model']
print("Best by F1:", best_name)

# yahan best_rf use kar rahe demo ke liye — apne winner ko plug karo
final_model = best_rf
y_pred = final_model.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4,3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No','Yes'], yticklabels=['No','Yes'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ROC curve
y_score = final_model.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_score)
plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_score):.3f}")
plt.plot([0,1],[0,1],'--', color='grey')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curve'); plt.legend()
plt.show()

## Conclusion
- Imbalanced healthcare data me **accuracy bekaar metric hai** — Recall (kitne actual strokes pakde) aur F1 dekho.
- `class_weight='balanced'` / SMOTE se minority class detect hoti hai.
- Tuning + ensembles se F1 stabilize hota hai.

**Next steps (resume/portfolio ke liye):**
1. Feature importance (RF) plot karke explainability add karo.
2. Threshold tuning — default 0.5 ki jagah recall-optimized threshold.
3. Best model ko FastAPI me wrap karke ek `/predict` endpoint bana do (yeh tere stack me fit hai).